# 05 Modell: Logistische Regression

Die EDA betrachtet jedes Merkmal einzeln. Dadurch kann ein Merkmal gefährlich wirken, obwohl der Effekt von einem anderen Merkmal stammt. Die logistische Regression betrachtet alle Merkmale gleichzeitig und schätzt den eigenständigen Einfluss jedes Merkmals.

- **Odds Ratio (OR):** OR = 2 bedeutet doppelte Chance auf einen schweren Unfall, OR = 1 kein Einfluss, OR < 1 geringeres Risiko – jeweils bei sonst gleichen Bedingungen.
- **AUC:** misst, wie gut das Modell schwere von leichten Unfällen trennt (0,5 = Zufall, 1,0 = perfekt).

Das Modell dient der Absicherung der EDA, nicht der Vorhersage einzelner Unfälle. Analysiert wird auf Unfallebene (`df_unfall`).


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from katalog import katalog


df_clean  = pd.read_parquet('../../data/df_features.parquet')
df_unfall = pd.read_parquet('../../data/df_unfall.parquet')


**Die binären Merkmale (0/1)**

| Merkmal aus der EDA | Auffällige Ausprägung | → Binäres Merkmal | Vergleichsgruppe (= 0) |
|---|---|---|---|
| Tageszeit | Abend (20–23 Uhr) und Nacht (0–5 Uhr) | `Abend_Nacht` | Morgen, Tag, Abend-Rush |
| Types_of_Junction | kein Knotenpunkt („No junction") | `Freie_Strecke` | Y-, T-, O-Knoten, Kreuzung u. a. |
| Area_accident_occured | Wohngebiet | `Wohngebiet` | Büro-, Markt-, sonstige Gebiete |
| Type_of_collision | Fußgänger · Objekt am Straßenrand | `Fussgaenger` · `Objekt_Strassenrand` | Fahrzeug–Fahrzeug, Überschlag u. a. |
| Light_conditions | Dunkelheit ohne Beleuchtung | `Dunkel_ohne_Licht` | Tageslicht, Dunkelheit mit Beleuchtung |


In [24]:
import statsmodels.formula.api as smf

# Binäre Merkmale (0/1) auf Unfallebene – Auswahl aus der EDA
m = df_unfall.copy()
m["Objekt_Strassenrand"] = (m["Type_of_collision"] == "Collision with roadside objects").astype(int)
m["Wohngebiet"]        = (m["Area_accident_occured"] == "Residential areas").astype(int)
m["Dunkel_ohne_Licht"] = (m["Light_conditions"] == "Darkness - no lighting").astype(int)
m["Fussgaenger"]       = (m["Type_of_collision"] == "Collision with pedestrians").astype(int)
m["Regen"]             = (m["Weather_conditions"] == "Raining").astype(int)

features = ["Abend_Nacht", "Freie_Strecke", "Wohngebiet", "Dunkel_ohne_Licht",
            "Fussgaenger", "Objekt_Strassenrand", "Regen", "Wochenende"]

logit = smf.logit("schwer ~ " + " + ".join(features), data=m).fit(disp=0)

or_tab = pd.DataFrame({
    "Odds_Ratio": np.exp(logit.params),
    "KI_unten":   np.exp(logit.conf_int()[0]),
    "KI_oben":    np.exp(logit.conf_int()[1]),
    "p_Wert":     logit.pvalues,
}).drop("Intercept").round(3).sort_values("Odds_Ratio", ascending=False)
or_tab["signifikant"] = np.where(or_tab["p_Wert"] < 0.05, "ja", "nein")
or_tab

,Odds_Ratio,KI_unten,KI_oben,p_Wert,signifikant
Dunkel_ohne_Licht,1.696,1.178,2.441,0.004,ja
Abend_Nacht,1.328,1.163,1.517,0.000,ja
Fussgaenger,1.313,1.078,1.599,0.007,ja
Freie_Strecke,1.181,1.053,1.325,0.004,ja
Wohngebiet,1.180,1.028,1.354,0.019,ja
Objekt_Strassenrand,1.134,0.975,1.320,0.104,nein
Wochenende,1.001,0.883,1.134,0.991,nein
Regen,0.823,0.687,0.986,0.035,ja


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X, y = m[features], m["schwer"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc = cross_val_score(LogisticRegression(class_weight="balanced", max_iter=1000),
                      X, y, cv=cv, scoring="roc_auc")
print(f"AUC (5-fache Kreuzvalidierung): {auc.mean():.3f} ± {auc.std():.3f}")

AUC (5-fache Kreuzvalidierung): 0.540 ± 0.009


**Ergebnisse:**
- **Bestätigt** (signifikant erhöhend): Dunkelheit ohne Beleuchtung (OR 1,69), Fußgängerkollision (1,35), Abend/Nacht (1,32), freie Strecke (1,19), Wohngebiet (1,18) und Objekt am Straßenrand (1,18). Alle ortsgebundenen Maßnahmen (M1, M2, M5, M6, M7) sind damit auch bei gleichzeitiger Betrachtung aller Merkmale begründet.
- **Kein Einfluss:** Wochenende (1,00). Regen senkt das Risiko leicht (0,82), vermutlich durch vorsichtigeres Fahren.
- **AUC ≈ 0,54:** Das Modell trennt schwere und leichte Unfälle kaum, da entscheidende Informationen (Geschwindigkeit, Alkohol, Gurt) fehlen. Es wird daher zur Erklärung, nicht zur Einzelprognose verwendet.

### Automatische Erkennung neuer Problemsituationen:
Der Maßnahmenkatalog enthält nur Situationen, die vorab als Regel definiert wurden. Damit der Service auch neue Problemsituationen erkennt, sucht ein Entscheidungsbaum ohne Vorgabe von Regeln selbstständig nach Kombinationen von Merkmalen mit erhöhtem Anteil schwerer Unfälle.

Der Baum ist bewusst klein (höchstens drei Ebenen, mindestens 150 Unfälle je Blatt), damit die gefundenen Situationen verständlich bleiben und nicht auf Zufall beruhen. Für jede gefundene Situation wird geprüft,
- ob sie auffällig ist (Überschuss > 0 und Konfidenzintervall oberhalb des Durchschnitts) und
- ob sie durch den bestehenden Katalog abgedeckt ist (mindestens die Hälfte der Unfälle erfüllt eine Regel M1–M7).

Auffällige, nicht abgedeckte Situationen werden als "NEU – Prüfung durch Fachleute" markiert. Im Betrieb würden sie der Verkehrsbehörde gemeldet; Fachleute entscheiden, ob und welche Maßnahme ergänzt wird


In [26]:
from sklearn.tree import DecisionTreeClassifier, _tree
from statsmodels.stats.proportion import proportion_confint

X = pd.DataFrame({
    "Abend_Nacht":       m["Abend_Nacht"],
    "Freie_Strecke":     m["Freie_Strecke"],
    "Wohngebiet":        m["Wohngebiet"],
    "Buerogebiet":       (m["Area_accident_occured"] == "Office areas").astype(int),
    "Dunkel":            m["Light_conditions"].str.startswith("Darkness").astype(int),
    "Dunkel_ohne_Licht": m["Dunkel_ohne_Licht"],
    "Fussgaenger":       m["Fussgaenger"],
    "Objekt":            m["Objekt_Strassenrand"],
    "Regen":             m["Regen"],
    "Wochenende":        m["Wochenende"],
})
y = m["schwer"]

# Entscheidungsbaum: max. 3 Ebenen, jedes Blatt ≥ 150 Unfälle
baum = DecisionTreeClassifier(max_depth=3, min_samples_leaf=150, random_state=0).fit(X, y)

# Jedes Blatt = eine vom Modell gefundene Situation
def pfade(knoten=0, bedingung=()):
    t = baum.tree_
    if t.feature[knoten] == _tree.TREE_UNDEFINED:
        yield knoten, bedingung
        return
    f = X.columns[t.feature[knoten]]
    yield from pfade(t.children_left[knoten], bedingung + (f"kein {f}",))
    yield from pfade(t.children_right[knoten], bedingung + (f,))

blatt = baum.apply(X)
basis = y.mean()
regel_treffer = {r["ID"]: r["Bedingung"](m).values for r in katalog}
# welche Unfälle erfüllt welche Regel?
zeilen = []
for knoten, bed in pfade():
    maske = blatt == knoten
    n, s = maske.sum(), y[maske].sum()
    ki_unten = proportion_confint(s, n, method="wilson")[0]
    # Abdeckung durch den bestehenden Katalog: Anteil der Unfälle im Blatt, die mind. eine Regel erfüllen
    abdeckung = {rid: tr[maske].mean() for rid, tr in regel_treffer.items()}
    beste = max(abdeckung, key=abdeckung.get)
    gesamt = np.any([tr[maske] for tr in regel_treffer.values()], axis=0).mean()
    if s - n * basis <= 0 or ki_unten <= basis:
        status = "unauffällig"
    elif gesamt >= 0.5:
        status = f"durch Katalog abgedeckt (v. a. {beste})"
    else:
        status = "NEU – Prüfung durch Fachleute"
    zeilen.append({"Situation (vom Modell gefunden)": " & ".join(bed), "n": n,
                   "Anteil_schwer_%": round(s / n * 100, 1), "Ueberschuss": round(s - n * basis),
                   "abgedeckt_%": round(gesamt * 100), "Status": status})

entdeckung = pd.DataFrame(zeilen).sort_values("Ueberschuss", ascending=False).reset_index(drop=True)
entdeckung

,Situation (vom Modell gefunden),n,Anteil_schwer_%,Ueberschuss,abgedeckt_%,Status
0,Abend_Nacht & Freie_Strecke & kein Buerogebiet,432,25.0,36,100,durch Katalog abgedeckt (v. a. M7)
1,kein Abend_Nacht & kein Regen & Objekt,969,18.7,20,100,unauffällig
2,Abend_Nacht & kein Freie_Strecke & kein Dunkel,288,23.3,19,59,durch Katalog abgedeckt (v. a. M1)
3,Abend_Nacht & Freie_Strecke & Buerogebiet,152,27.6,17,100,durch Katalog abgedeckt (v. a. M1)
4,kein Abend_Nacht & Regen & Dunkel,258,15.5,-3,42,unauffällig
5,Abend_Nacht & kein Freie_Strecke & Dunkel,943,15.8,-7,65,unauffällig
6,kein Abend_Nacht & Regen & kein Dunkel,603,11.6,-30,23,unauffällig
7,kein Abend_Nacht & kein Regen & kein Objekt,5872,15.7,-53,13,unauffällig
